# Aing_리그전 ResNet (CIFAR-10)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aing-gachon/26-Spring-ResNet-Study/blob/main/Week%203/Aing_league_ResNet_CIFAR10.ipynb)

> 목표: **ResNet 구조(모델 하이퍼파라미터)** 를 바꿔가며 성능 변화를 비교하고, Residual/Shortcut 개념을 코드로 체험합니다.  
> 최종 점수(리그전): **CIFAR-10 test accuracy** *(best valid checkpoint 기준)*

---

## 리그전 규칙

### FIXED (공정성/평가 계약 — 변경 금지)
- 데이터: **CIFAR-10** (`uoft-cs/cifar10`)
- Split/Subsample: **Train 10,000 / Valid 2,000**, seed=42 *(train에서 고정 추출)*
- 학습 예산: **EPOCHS 고정** *(epoch 기반)*
  - **권장(120분 제한 기준): 25 epochs**
- 배치(유효 배치 고정): **batch=128**
  - OOM이면 **batch=64 + grad_accum=2** 조합만 허용
- Metric: **Accuracy**
- Final/Test: **동일한 데이터/전처리 계약**으로 `test` split에서 평가 *(best valid checkpoint 로드 후)*

### TUNE (명시된 부분 step 5 & 6 부분만 수정 가능)
- ex)
  - Augmentation / Normalization 구현(단, test는 deterministic 권장)
  - Optimizer/Scheduler, loss, AMP, regularization 등
  - Logging/visualization, 체크포인트 저장 방식 등

---

**속도 참고(환경에 따라 상이)**  
- 10k train subset 기준, 25 epochs는 보통 “수 분 ~ 수십 분” 범위에서 끝나도록 설계했습니다.  


### STEP1. 환경 확인 (선택)
- Colab은 보통 torch/torchvision이 이미 설치되어 있습니다.

In [ ]:
# (선택) 설치가 필요할 때만 실행하세요.
#!pip -q install -U torch torchvision datasets numpy matplotlib

import torch, torchvision, numpy as np
import matplotlib.pyplot as plt

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("numpy:", np.__version__)

### STEP2. seed / device / warning 정리 
### ! device가 CPU로 나온다면 런타임 유형을 확인해보세요
- 실험 재현성을 위해 seed를 고정합니다.
- 노이즈성 warning(라이브러리 deprecation)을 최소화합니다.

In [ ]:
import os, json, time, random, math, warnings
from dataclasses import dataclass
from typing import List, Type, Dict, Any, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T

# ---- warning 최소화 (학습/평가 자체에는 영향 없음) ----
warnings.filterwarnings(
    "ignore",
    message="dtype\(\): align should be passed as Python or NumPy boolean",
)
# torch.cuda.amp deprecation warning은 아래에서 torch.amp API로 해결(따로 ignore 필요 없음)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # 입력 크기가 고정(32x32)이라 보통 속도 이점이 있음
    torch.backends.cudnn.benchmark = True

SEED_FIXED = 42
set_seed(SEED_FIXED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

from datetime import datetime
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = f"./runs/resnet_league_cifar10_{RUN_TAG}"
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

### STEP3. CIFAR-10 로드 + 고정 Split/Subsample (B preset)
- **기본 경로:** `datasets.load_dataset("uoft-cs/cifar10")`로 **Hugging Face Datasets에서 로드**합니다 (가능하면 캐시 사용).
- **HF Hub 접근이 막힌 환경**이라면 로컬 `/data` 에서 오프라인으로 로드합니다.  
  - 로컬 경로는 기본값이 `/data`이며, 필요하면 환경변수 `CIFAR10_ROOT`로 변경할 수 있습니다.


In [ ]:
# === (고정) 데이터 고정값 ===
TRAIN_SAMPLES_FIXED = 10_000
VALID_SAMPLES_FIXED = 2_000
SPLIT_SEED_FIXED = 42


# === (공정성) 리그전 규칙(런타임 검증) ===
EPOCHS_FIXED = 25  # FIXED: epoch budget (권장: 25 epochs)

def validate_league_rules(batch_size: int, grad_accum: int, train_samples: int, valid_samples: int, split_seed: int, epochs: int):
    # dataset/split 계약
    assert train_samples == TRAIN_SAMPLES_FIXED, "FIXED 위반: train_samples"
    assert valid_samples == VALID_SAMPLES_FIXED, "FIXED 위반: valid_samples"
    assert split_seed == SPLIT_SEED_FIXED, "FIXED 위반: split_seed"
    assert epochs == EPOCHS_FIXED, "FIXED 위반: epochs"

    # 유효 배치 계약
    ok = (batch_size == 128 and grad_accum == 1) or (batch_size == 64 and grad_accum == 2)
    assert ok, "FIXED 위반: batch 규칙은 (128,accum1) 또는 (64,accum2)만 허용"

BATCH_SIZE_FIXED = 128
NUM_WORKERS_FIXED = 2
PIN_MEMORY_FIXED = True

# CIFAR-10 normalize (널리 쓰는 값)
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

def build_transforms():
    train_tf = T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ])
    test_tf = T.Compose([
        T.ToTensor(),
        T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ])
    return train_tf, test_tf

class _HFCIFAR10(torch.utils.data.Dataset):
    def __init__(self, hf_split, transform=None):
        self.ds = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        img = item["img"]  # PIL.Image
        y = int(item["label"])
        if self.transform is not None:
            img = self.transform(img)
        return img, y

def build_cifar10_dataloaders(
    train_samples: int,
    valid_samples: int,
    split_seed: int,
    batch_size: int,
    num_workers: int,
):
    train_tf, test_tf = build_transforms()

    # 같은 raw train split을, transform만 다르게 2개 로딩(aug on/off)
    # 1) 기본 경로: Hugging Face Datasets (uoft-cs/cifar10)
    # 2) HF Hub 접근이 막혀있으면: 로컬 오프라인 데이터셋(/data)로 진행 (download=False)
    #    - /data 아래에 torchvision CIFAR10이 읽을 수 있는 형태로 준비되어 있어야 합니다.
    #    - 로컬에도 없으면: "다운받아서 /data에 넣고 다시 실행" 하도록 에러로 중단합니다.
    try:
        from datasets import load_dataset

        hf = load_dataset("uoft-cs/cifar10")
        train_hf = hf["train"]
        test_hf = hf["test"]

        train_base = _HFCIFAR10(train_hf, transform=train_tf)
        valid_base = _HFCIFAR10(train_hf, transform=test_tf)
        test_set = _HFCIFAR10(test_hf, transform=test_tf)

    except Exception as e:
        print("[WARN] HF dataset load failed -> try local /data (offline) :", repr(e))

        local_root = os.environ.get("CIFAR10_ROOT", "/data")
        if not os.path.isdir(local_root):
            raise RuntimeError(
                f"""Hugging Face 접근이 막혔고, 로컬 데이터 경로도 없습니다.
                    - 기대 경로: {local_root} (디렉토리)
                    - 해결: CIFAR-10을 미리 다운로드/압축해제해서 위 경로에 넣고 다시 실행하세요.
                    (또는 환경변수 CIFAR10_ROOT로 경로를 지정할 수 있습니다.)"""
            ) from e

        try:
            # ⚠️ download=False: 네트워크 없이 로컬 파일만 사용
            train_base = torchvision.datasets.CIFAR10(local_root, train=True, download=False, transform=train_tf)
            valid_base = torchvision.datasets.CIFAR10(local_root, train=True, download=False, transform=test_tf)
            test_set = torchvision.datasets.CIFAR10(local_root, train=False, download=False, transform=test_tf)
        except Exception as e2:
            raise RuntimeError(
                f"""로컬 CIFAR-10을 찾지 못했습니다.
                    - root: {local_root}
                    - torchvision CIFAR10이 읽을 수 있도록 root 아래에 'cifar-10-batches-py' 폴더가 있어야 합니다.
                    - 해결: CIFAR-10 python 버전을 다운로드 후 압축 해제하여 root 아래에 두세요."""
            ) from e2

    n = len(train_base)
    assert train_samples + valid_samples <= n, "train+valid 샘플 수가 CIFAR-10 train(50k)을 넘습니다."

    rng = np.random.default_rng(split_seed)
    idx = np.arange(n)
    rng.shuffle(idx)

    train_idx = idx[:train_samples]
    valid_idx = idx[train_samples:train_samples + valid_samples]

    train_set = torch.utils.data.Subset(train_base, train_idx.tolist())
    valid_set = torch.utils.data.Subset(valid_base, valid_idx.tolist())

    g = torch.Generator()
    g.manual_seed(split_seed)

    train_loader = torch.utils.data.DataLoader(
        train_set,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=PIN_MEMORY_FIXED,
        persistent_workers=(num_workers > 0),
        generator=g,
    )
    valid_loader = torch.utils.data.DataLoader(
        valid_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=PIN_MEMORY_FIXED,
        persistent_workers=(num_workers > 0),
    )
    test_loader = torch.utils.data.DataLoader(
        test_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=PIN_MEMORY_FIXED,
        persistent_workers=(num_workers > 0),
    )
    return train_loader, valid_loader, test_loader

train_loader, valid_loader, test_loader = build_cifar10_dataloaders(
    train_samples=TRAIN_SAMPLES_FIXED,
    valid_samples=VALID_SAMPLES_FIXED,
    split_seed=SPLIT_SEED_FIXED,
    batch_size=BATCH_SIZE_FIXED,
    num_workers=NUM_WORKERS_FIXED,
)

print("train steps/epoch (approx):", math.ceil(TRAIN_SAMPLES_FIXED / BATCH_SIZE_FIXED))
print("train samples:", TRAIN_SAMPLES_FIXED)
print("valid samples:", VALID_SAMPLES_FIXED)
print("test samples :", len(test_loader.dataset))


### STEP4. 모델 구현 (빈칸 코드와 동일한 구조)

포인트:
- `ShortcutZeroPad`: stride downsample + channel zero-pad (Option A)
- `ShortcutProjection`: 1×1 conv + BN (Option B)
- `BasicBlockV1`: ResNet v1(post-activation)
- `ResNetV1`: CIFAR/Imagenet stem 모두 지원 (리그전은 CIFAR만 사용)
- `resnet_cifar`: depth=6n+2 규칙
- `resnet_imagenet`: 참고용(빈칸 코드 appendix)

In [ ]:
# =========================================================
# 1) Shortcut (Option A: zero-pad, Option B: projection)
# =========================================================

class ShortcutZeroPad(nn.Module):
    """
    Option A (CIFAR): 학습 파라미터 없이 차원만 맞추는 shortcut.
    - stage 경계에서 (H,W)가 줄어들거나 채널이 늘어날 때도 F(x) + shortcut(x)가 가능하도록
      shortcut 경로의 shape를 맞춥니다.
    """

    def __init__(self, in_channels: int, out_channels: int, stride: int):
        super().__init__()
        if out_channels < in_channels:
            raise ValueError(
                "ShortcutZeroPad only supports channel increase (out_channels >= in_channels). "
                "Use projection shortcut (1x1 conv) for channel decrease."
            )
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (H,W) 맞추기
        if self.stride != 1:
            x = x[:, :, ::self.stride, ::self.stride]

        # (C) 맞추기
        pad_channels = self.out_channels - self.in_channels
        if pad_channels == 0:
            return x

        zeros = torch.zeros(
            (x.size(0), pad_channels, x.size(2), x.size(3)),
            device=x.device,
            dtype=x.dtype,
        )

        # channel 축 concat
        return torch.cat([x, zeros], dim=1)

class ShortcutProjection(nn.Module):
    """
    Option B: 1×1 conv(+BN)로 shortcut 경로도 학습 가능하게 만들어 차원을 맞춥니다.
    - 차원 불일치가 생기는 지점에서 W_s x 형태로 변환한 뒤 더할 수 있게 합니다.
    """

    def __init__(self, in_channels: int, out_channels: int, stride: int):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            stride=stride,
            padding=0,
            bias=False,
        )
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.bn(self.conv(x))

# =========================================================
# 2) Residual Block (ResNet v1: post-activation)
# =========================================================

class BasicBlockV1(nn.Module):
    """
    Figure 2 (left): conv-bn-relu -> conv-bn -> add -> relu  (ResNet v1)
    """
    expansion = 1

    def __init__(self, in_channels: int, out_channels: int, stride: int, shortcut: str = "projection"):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        if stride == 1 and in_channels == out_channels:
            self.shortcut = nn.Identity()
        else:
            if shortcut == "projection":
                self.shortcut = ShortcutProjection(in_channels, out_channels, stride=stride)
            elif shortcut == "zero_pad":
                self.shortcut = ShortcutZeroPad(in_channels, out_channels, stride=stride)
            else:
                raise ValueError(f"Unknown shortcut mode: {shortcut}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x)

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out = out + identity
        out = self.relu(out)
        return out

# =========================================================
# 3) (심화) BottleneckV1 (appendix)
# =========================================================

class BottleneckV1(nn.Module):
    """
    Figure 5 (right) bottleneck (ImageNet용에서 주로 사용).
    이 노트북에서는 심화(선택) 섹션으로만 다룹니다.
    """
    expansion = 4

    def __init__(self, in_channels: int, planes: int, stride: int, shortcut: str = "projection"):
        super().__init__()
        out_channels = planes * self.expansion

        self.conv1 = nn.Conv2d(in_channels, planes, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)

        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.conv3 = nn.Conv2d(planes, out_channels, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        if stride == 1 and in_channels == out_channels:
            self.shortcut = nn.Identity()
        else:
            if shortcut == "projection":
                self.shortcut = ShortcutProjection(in_channels, out_channels, stride=stride)
            elif shortcut == "zero_pad":
                self.shortcut = ShortcutZeroPad(in_channels, out_channels, stride=stride)
            else:
                raise ValueError(f"Unknown shortcut mode: {shortcut}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x)

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        out = out + identity
        out = self.relu(out)
        return out

# =========================================================
# 4) ResNetV1 (cifar / imagenet)
# =========================================================

class ResNetV1(nn.Module):
    def __init__(
        self,
        block: Type[nn.Module],
        layers: List[int],
        num_classes: int,
        stem: str,        # "imagenet" or "cifar"
        shortcut: str,    # "projection" or "zero_pad"
        cifar_base_channels: int = 16,
    ):
        super().__init__()
        self.shortcut = shortcut

        if stem == "imagenet":
            self.in_channels = 64
            self.stem = nn.Sequential(
                nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            )
            stage_planes = [64, 128, 256, 512]
        elif stem == "cifar":
            self.in_channels = cifar_base_channels
            self.stem = nn.Sequential(
                nn.Conv2d(3, cifar_base_channels, kernel_size=3, stride=1, padding=1, bias=False),
                nn.BatchNorm2d(cifar_base_channels),
                nn.ReLU(inplace=True),
            )
            stage_planes = [cifar_base_channels, cifar_base_channels * 2, cifar_base_channels * 4]
        else:
            raise ValueError("stem must be 'imagenet' or 'cifar'")

        self.layer1 = self._make_layer(block, stage_planes[0], layers[0], stride=1)
        self.layer2 = self._make_layer(block, stage_planes[1], layers[1], stride=2)
        self.layer3 = self._make_layer(block, stage_planes[2], layers[2], stride=2)

        if stem == "imagenet":
            self.layer4 = self._make_layer(block, stage_planes[3], layers[3], stride=2)
            final_ch = stage_planes[3] * getattr(block, "expansion", 1)
        else:
            self.layer4 = None
            final_ch = stage_planes[2] * getattr(block, "expansion", 1)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(final_ch, num_classes)

    def _make_layer(self, block: Type[nn.Module], planes: int, blocks: int, stride: int) -> nn.Sequential:
        layers = []
        expansion = getattr(block, "expansion", 1)

        if block is BottleneckV1:
            layers.append(block(self.in_channels, planes, stride=stride, shortcut=self.shortcut))
            self.in_channels = planes * expansion
            for _ in range(1, blocks):
                layers.append(block(self.in_channels, planes, stride=1, shortcut=self.shortcut))
        else:
            out_channels = planes * expansion
            layers.append(block(self.in_channels, out_channels, stride=stride, shortcut=self.shortcut))
            self.in_channels = out_channels
            for _ in range(1, blocks):
                layers.append(block(self.in_channels, out_channels, stride=1, shortcut=self.shortcut))

        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        if self.layer4 is not None:
            x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

# =========================================================
# 5) Factory (CIFAR: depth = 6n + 2)
# =========================================================

def resnet_cifar(depth: int, num_classes: int = 10, shortcut: str = "zero_pad", base_channels: int = 16):
    if (depth - 2) % 6 != 0:
        raise ValueError("CIFAR depth must be 6n+2")
    n = (depth - 2) // 6
    return ResNetV1(BasicBlockV1, [n, n, n], num_classes, stem="cifar", shortcut=shortcut, cifar_base_channels=base_channels)

# Appendix: ImageNet 용 (참고)
def resnet_imagenet(depth: int, num_classes: int = 1000, shortcut: str = "projection"):
    if depth == 18:
        return ResNetV1(BasicBlockV1, [2, 2, 2, 2], num_classes, stem="imagenet", shortcut=shortcut)
    if depth == 34:
        return ResNetV1(BasicBlockV1, [3, 4, 6, 3], num_classes, stem="imagenet", shortcut=shortcut)
    if depth == 50:
        return ResNetV1(BottleneckV1, [3, 4, 6, 3], num_classes, stem="imagenet", shortcut=shortcut)
    if depth == 101:
        return ResNetV1(BottleneckV1, [3, 4, 23, 3], num_classes, stem="imagenet", shortcut=shortcut)
    if depth == 152:
        return ResNetV1(BottleneckV1, [3, 8, 36, 3], num_classes, stem="imagenet", shortcut=shortcut)
    raise ValueError("Unsupported depth for ImageNet ResNet")

# quick check
with torch.no_grad():
    _m = resnet_cifar(depth=20, num_classes=10, shortcut="zero_pad", base_channels=16)
    _y = _m(torch.randn(4, 3, 32, 32))
print("[OK] forward:", tuple(_y.shape))

### STEP5. MODEL_HP (모델 구조 관련 설정, TUNE)
- 설명: 이 단계는 ResNet 아키텍처의 핵심 하이퍼파라미터를 조정합니다.
  - `depth`: `6n+2` 규칙에 따라 블록 깊이를 변경하여 모델 용량 및 표현력 제어
  - `shortcut`: `zero_pad`(파라미터 없음, 가볍고 빠름) 또는 `projection`(1x1 conv + BN, 더 유연) 선택
  - `base_channels`: 각 stage 시작 채널 수 (기본 16, 확장 24 등)

In [ ]:
# ===  TUNE: 이 값만 바꿔서 실험하세요 ===
MODEL_HP = dict(
    depth=56,              # TUNE: 20 / 32 / 56
    shortcut="zero_pad",   # TUNE: "zero_pad" / "projection"
    base_channels=16,      # TUNE: 16 / 24
)

def build_model(hp: Dict[str, Any]) -> nn.Module:
    return resnet_cifar(
        depth=int(hp["depth"]),
        num_classes=10,
        shortcut=str(hp["shortcut"]),
        base_channels=int(hp["base_channels"]),
    )

model = build_model(MODEL_HP).to(device)

n_params = sum(p.numel() for p in model.parameters())
print("MODEL_HP:", MODEL_HP)
print("params(M):", n_params / 1e6)

### STEP6. 학습 설정 + train/eval
- **FIXED(공정성)**: epochs / split / 유효 배치 / metric / final test 규칙만 유지합니다.
- 그 외(optimizer, scheduler, aug 등)는 모범 코드 목적상 **원하면 자유롭게 수정** 가능합니다.
- 학습 프로세스는 조정 가능합니다.
  - `LR`, `WEIGHT_DECAY`, `MOMENTUM`: optimizer 하이퍼파라미터 튜닝
  - `Optimizer` / `Scheduler` 선택: 기본 SGD + CosineAnnealingLR 외에도 Adam/Warmup/Cosine 등 가능
  - `GRAD_ACCUM`: 배치 규칙(128/1 또는 64/2) 준수하면서 효과적인 batch 크기 확대
  - `Augmentation`: train transform을 변경해 일반화 성능 개선

In [ ]:
# === 학습/평가 설정 ===
# FIXED: epoch budget은 STEP5에서 정의된 EPOCHS_FIXED를 사용합니다.
# FIXED: batch 규칙은 validate_league_rules로 강제합니다.

# (TUNE) 아래 값들은 기본값이며 필요하면 바꿔도 됩니다.
LR = 0.1
WEIGHT_DECAY = 1e-4
MOMENTUM = 0.9

# FIXED 계약: (batch, grad_accum) 조합만 허용
GRAD_ACCUM = 1  # OOM이면 2로 바꾸고, 그때 batch=64로만 허용

# FIXED AMP 사용 여부
# 리그전에서 고정값으로 설정 (float16 mixed precision을 무조건 사용하지 않음)
AMP = False

# (TUNE) Loss Function (Regularization 추가 여부)
criterion = nn.CrossEntropyLoss()

# (Tune) 원하는 Optimizer 사용 가능
optimizer = optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)

# epoch 기반 cosine
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FIXED)

def get_amp(device: torch.device, enabled: bool):
    if enabled and device.type == "cuda":
        scaler = torch.amp.GradScaler("cuda", enabled=True)
        autocast_ctx = lambda: torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=True)
        return scaler, autocast_ctx
    else:
        scaler = None
        autocast_ctx = lambda: torch.amp.autocast(device_type="cpu", enabled=False)
        return scaler, autocast_ctx

@torch.no_grad()
def evaluate(model: nn.Module, loader, device: torch.device) -> Dict[str, float]:
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)

        total_loss += float(loss.item()) * x.size(0)
        pred = logits.argmax(dim=1)
        correct += int((pred == y).sum().item())
        total += int(y.size(0))

    return {"loss": total_loss / max(total, 1), "acc": correct / max(total, 1)}

def train_epochs(
    model: nn.Module,
    train_loader,
    valid_loader,
    device: torch.device,
    epochs: int,
    grad_accum: int,
    amp: bool,
):
    scaler, autocast_ctx = get_amp(device, amp)

    global_step = 0
    best_valid_acc = -1.0
    best_epoch = -1
    best_state = None
    best_valid_at_best = None  # {"loss","acc"}

    history = {
        "epoch": [],
        "global_step": [],
        "train_loss": [],
        "train_acc": [],
        "valid_loss": [],
        "valid_acc": [],
        "lr": [],
    }

    start_time = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        running_total = 0

        optimizer.zero_grad(set_to_none=True)

        for it, (x, y) in enumerate(train_loader, start=1):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            if scaler is not None:
                with autocast_ctx():
                    logits = model(x)
                    loss = criterion(logits, y) / grad_accum
                scaler.scale(loss).backward()
            else:
                logits = model(x)
                loss = criterion(logits, y) / grad_accum
                loss.backward()

            # stats (원래 loss 스케일로 환산)
            running_loss += float(loss.item()) * grad_accum * x.size(0)
            pred = logits.argmax(dim=1)
            running_correct += int((pred == y).sum().item())
            running_total += int(y.size(0))

            # optimizer step (accum)
            if it % grad_accum == 0:
                if scaler is not None:
                    scale_before = float(scaler.get_scale())
                    scaler.step(optimizer)
                    scaler.update()
                    scale_after = float(scaler.get_scale())
                    if scale_after >= scale_before:
                        pass  # epoch 기반 scheduler는 epoch 끝에서만 step
                else:
                    optimizer.step()

                optimizer.zero_grad(set_to_none=True)

            global_step += 1

        # epoch end scheduler
        scheduler.step()

        train_loss = running_loss / max(running_total, 1)
        train_acc = running_correct / max(running_total, 1)

        valid = evaluate(model, valid_loader, device)

        history["epoch"].append(epoch)
        history["global_step"].append(global_step)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["valid_loss"].append(valid["loss"])
        history["valid_acc"].append(valid["acc"])
        history["lr"].append(float(optimizer.param_groups[0]["lr"]))

        # best checkpoint by valid acc
        if valid["acc"] > best_valid_acc:
            best_valid_acc = float(valid["acc"])
            best_epoch = epoch
            best_valid_at_best = {"loss": float(valid["loss"]), "acc": float(valid["acc"])}
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"[epoch {epoch:03d}/{epochs}] "
            f"train loss={train_loss:.4f} acc={train_acc*100:.2f}% | "
            f"valid loss={valid['loss']:.4f} acc={valid['acc']*100:.2f}% | "
            f"lr={optimizer.param_groups[0]['lr']:.6f}"
        )

    train_wall_time_sec = time.time() - start_time
    return {
        "history": history,
        "best_state": best_state,
        "best_epoch": best_epoch,
        "final_epoch": epochs,
        "train_wall_time_sec": train_wall_time_sec,
        "best_valid_at_best": best_valid_at_best,
    }


### STEP7. 학습 실행 + 최종 스코어보드 출력
- 리그전 최종 점수는 **CIFAR-10 test acc** 입니다.
- 단, 튜닝/선택은 valid 기준 best checkpoint로 합니다.

In [ ]:
# === (FIXED) 규칙 검증 ===
validate_league_rules(
    batch_size=BATCH_SIZE_FIXED,
    grad_accum=GRAD_ACCUM,
    train_samples=TRAIN_SAMPLES_FIXED,
    valid_samples=VALID_SAMPLES_FIXED,
    split_seed=SPLIT_SEED_FIXED,
    epochs=EPOCHS_FIXED,
)

result = train_epochs(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    device=device,
    epochs=EPOCHS_FIXED,
    grad_accum=GRAD_ACCUM,
    amp=AMP,
)

history = result["history"]
best_state = result["best_state"]
if best_state is None:
    raise RuntimeError("best_state가 없습니다. 학습/평가 루프를 확인하세요.")

# best weights load
model.load_state_dict(best_state, strict=True)

best_valid = evaluate(model, valid_loader, device)
best_test  = evaluate(model, test_loader, device)

FINAL_VALID_LOSS = best_valid["loss"]
FINAL_VALID_ACC  = best_valid["acc"]
FINAL_TEST_LOSS  = best_test["loss"]
FINAL_TEST_ACC   = best_test["acc"]

BEST_VALID_AT_BEST = result.get("best_valid_at_best") or {"loss": None, "acc": None}

print("\n" + "="*60)
print("🏁 FINAL SCOREBOARD (best-valid checkpoint)")
print("="*60)
print(f"MODEL_HP: {MODEL_HP}")
print(f"params(M): {sum(p.numel() for p in model.parameters())/1e6:.6f}")
print(f"best_epoch: {result['best_epoch']} / final_epoch: {result['final_epoch']}")
print(f"train_wall_time_sec: {result['train_wall_time_sec']:.1f}s")
print("-"*60)
print(f"[best(valid) during train]  loss={BEST_VALID_AT_BEST['loss']:.4f}  acc={BEST_VALID_AT_BEST['acc']*100:.2f}%")
print(f"[after load best] valid    loss={FINAL_VALID_LOSS:.4f}  acc={FINAL_VALID_ACC*100:.2f}%")
print(f"[after load best] test     loss={FINAL_TEST_LOSS:.4f}  acc={FINAL_TEST_ACC*100:.2f}%")
print("-"*60)
print("LEAGUE_FINAL_SCORE_CIFAR10_TEST_ACC:", FINAL_TEST_ACC)
print("="*60)

# save checkpoint + summary
ckpt_path = os.path.join(OUT_DIR, "best_model.pt")
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "MODEL_HP": MODEL_HP,
        "metrics": {
            "best_epoch": result["best_epoch"],
            "best_valid_loss_during_train": BEST_VALID_AT_BEST["loss"],
            "best_valid_acc_during_train": BEST_VALID_AT_BEST["acc"],
            "final_valid_loss": FINAL_VALID_LOSS,
            "final_valid_acc": FINAL_VALID_ACC,
            "final_test_loss": FINAL_TEST_LOSS,
            "final_test_acc": FINAL_TEST_ACC,
        },
        "rules_fixed": {
            "dataset": "cifar10",
            "train_samples": TRAIN_SAMPLES_FIXED,
            "valid_samples": VALID_SAMPLES_FIXED,
            "split_seed": SPLIT_SEED_FIXED,
            "epochs": EPOCHS_FIXED,
            "batch_rule": "batch=128 or (batch=64,accum=2)",
            "metric": "accuracy",
        },
    },
    ckpt_path,
)
print(f"saved: {ckpt_path}")


### (추가) 로그 곡선 시각화
- train/valid 곡선을 눈으로 확인합니다.

In [ ]:
import matplotlib.pyplot as plt

if len(history["step"]) > 0:
    plt.figure()
    plt.plot(history["step"], history["train_loss"], marker="o")
    plt.title("Train loss (logged)")
    plt.xlabel("step"); plt.ylabel("loss")
    plt.grid(True, alpha=0.3)
    plt.show()

    plt.figure()
    plt.plot(history["step"], [a*100 for a in history["train_acc"]], marker="o")
    plt.title("Train accuracy (logged)")
    plt.xlabel("step"); plt.ylabel("acc (%)")
    plt.grid(True, alpha=0.3)
    plt.show()

if len(history["valid_step"]) > 0:
    plt.figure()
    plt.plot(history["valid_step"], [a*100 for a in history["valid_acc"]], marker="o")
    plt.title("Valid accuracy (during train)")
    plt.xlabel("step"); plt.ylabel("acc (%)")
    plt.grid(True, alpha=0.3)
    plt.show()

### (추가) 예측 예시 시각화 (test 일부)
- 모델이 실제로 어떤 이미지를 어떻게 분류하는지 확인합니다.

In [ ]:
CLASSES = ("airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck")

@torch.no_grad()
def show_predictions(model: nn.Module, loader, device: torch.device, k: int = 8):
    model.eval()
    x, y = next(iter(loader))
    x = x.to(device, non_blocking=True)
    logits = model(x)
    pred = logits.argmax(dim=1).cpu()

    # denormalize for display
    x_cpu = x.cpu()
    mean = torch.tensor(CIFAR10_MEAN).view(1,3,1,1)
    std  = torch.tensor(CIFAR10_STD).view(1,3,1,1)
    x_vis = (x_cpu * std + mean).clamp(0,1)

    k = min(k, x_vis.size(0))
    plt.figure(figsize=(12, 3))
    for i in range(k):
        plt.subplot(1, k, i+1)
        plt.imshow(np.transpose(x_vis[i].numpy(), (1,2,0)))
        plt.title(f"P:{CLASSES[int(pred[i])]}\nT:{CLASSES[int(y[i])]}", fontsize=9)
        plt.axis("off")
    plt.show()

show_predictions(model, test_loader, device, k=8)